In [1]:
from environment import *
from movement import *


import numpy as np
import pandas as pd

from tqdm import tqdm

import pickle
import math
import visual

pygame 2.1.0 (SDL 2.0.16, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
# with open("Q.pkl", "rb") as f:
#     Q = pickle.load(f)

In [ ]:
# with open("all_positions.pkl", "rb") as f:
#     all_positions = pickle.load(f)

In [2]:
def get_actions(combined_action):
    action1 = math.floor(combined_action / 4)
    action2 = combined_action % 4
    return All_Actions(action1, action2)


In [3]:
Q = {}

all_positions = []
all_rewards = []
env = GridWorld_Portal()

Q[env.all_entity_positions] = [200] * 16

In [4]:
all_positions = []
all_rewards = []

In [5]:
def epsilon_greedy(state, epsilon):

    if np.random.random() < epsilon:
        return np.random.randint(16)
    else:
        combined_a = np.argmax(Q[state]).item()
        return combined_a

In [6]:
def update_Q(previous_state, action, next_state, reward):


    next_pos1, next_pos2, _ = next_state.get_positions()

    if next_state not in Q and (next_pos1.get_position()[0] > 9 or next_pos2.get_position()[0] > 9):
        Q[next_state] = [5000] * 16
    elif next_state not in Q:
        Q[next_state] = [200] * 16

    Q[previous_state][action] = Q[previous_state][action] + 0.1 * (
        reward + 0.99 * max(Q[next_state]) - Q[previous_state][action]
    )

In [7]:
def get_reward(previous_state, next_state):

    reward = 0
    _, _, box_pos = previous_state.get_positions()
    next_pos1, next_pos2, next_box_pos = next_state.get_positions()

    if next_state not in Q:
        reward += 1

    if box_pos != next_box_pos:
        reward += 2

    if next_pos1.get_position()[0] > 9:
        reward += 3

    if next_pos2.get_position()[0] > 9:
        reward += 3

    if next_pos1.get_position()[0] >= 19:
        reward += 4
    if next_pos2.get_position()[0] >= 19:
        reward += 4


    return reward

In [9]:
for i in tqdm(range(120000)):

    env.reset()
    rewards = [0] * 1000
    positions = [0] * 1000

    for j in range(1000):
        previous_state = env.all_entity_positions
        positions[j] = previous_state

        combined_action = epsilon_greedy(previous_state, 0.2)
        actions = get_actions(combined_action)

        next_state = env.step(actions)

        reward = get_reward(previous_state, next_state)

        rewards[j] = reward

        update_Q(previous_state, combined_action, next_state, reward)



    if i%1000 == 0:
        all_positions.append(positions)
        all_rewards.append(rewards)


100%|██████████| 120000/120000 [1:45:27<00:00, 18.96it/s] 


In [13]:
# Change this index to view which iteration do you want to see. We see iterations in multiples of 1000s.
training_run_index = -1

positions = all_positions[training_run_index]

In [17]:
positions = [position.get_positions() for position in positions]

In [18]:
visual.player1_positions = [position[0].get_position() for position in positions]
visual.player2_positions = [position[1].get_position() for position in positions]
visual.movable_object = [position[2].get_position() for position in positions]

In [23]:
game = visual.Game()
game.game_loop()

In [ ]:
# To build video
# Video().build_video("office_and_garden_output14", game.frames)

In [76]:
def show_value_counts(all_rewards):
    for rewards in all_rewards:
        print(pd.Series(rewards).value_counts())

In [24]:
# Save
with open("Q.pkl", "wb") as f:
    pickle.dump(Q, f)

In [25]:
with open("all_positions.pkl", "wb") as f:
    pickle.dump(all_positions, f)